In [1]:
from __future__ import annotations

import os
from datetime import UTC, datetime
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "LearningMilestone"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

milestones = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="difficulty",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="tags",
            data_type=wvc.config.DataType.TEXT_ARRAY,
        ),
        wvc.config.Property(
            name="completed_at",
            data_type=wvc.config.DataType.DATE,
        ),
        wvc.config.Property(
            name="duration_minutes",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="confidence_score",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="production_ready",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: LearningMilestone


In [5]:
def dt(year: int, month: int, day: int) -> datetime:
    return datetime(year, month, day, tzinfo=UTC)

In [6]:
milestone_data = [
    {
        "title": "Local Docker Weaviate",
        "description": "Started Weaviate locally in Docker and connected from a Python notebook.",
        "topic": "docker",
        "difficulty": "beginner",
        "tags": ["docker", "local", "setup"],
        "completed_at": dt(2026, 5, 1),
        "duration_minutes": 45,
        "confidence_score": 7.8,
        "production_ready": False,
    },
    {
        "title": "Weaviate Cloud Connection",
        "description": "Connected to Weaviate Cloud using cluster URL, Weaviate API key and OpenAI provider header.",
        "topic": "cloud",
        "difficulty": "beginner",
        "tags": ["cloud", "auth", "openai"],
        "completed_at": dt(2026, 5, 3),
        "duration_minutes": 35,
        "confidence_score": 8.4,
        "production_ready": True,
    },
    {
        "title": "CRUD with UUIDs",
        "description": "Created, fetched, updated and deleted Weaviate objects using UUID identifiers.",
        "topic": "crud",
        "difficulty": "beginner",
        "tags": ["crud", "uuid", "objects"],
        "completed_at": dt(2026, 5, 5),
        "duration_minutes": 50,
        "confidence_score": 8.7,
        "production_ready": True,
    },
    {
        "title": "Vector Search",
        "description": "Used near_text queries to search objects by semantic meaning.",
        "topic": "vector_search",
        "difficulty": "intermediate",
        "tags": ["vectors", "embeddings", "semantic-search"],
        "completed_at": dt(2026, 5, 7),
        "duration_minutes": 60,
        "confidence_score": 9.0,
        "production_ready": True,
    },
    {
        "title": "Hybrid Search",
        "description": "Combined BM25 keyword search with vector search using alpha.",
        "topic": "hybrid_search",
        "difficulty": "intermediate",
        "tags": ["bm25", "hybrid", "semantic-search"],
        "completed_at": dt(2026, 5, 9),
        "duration_minutes": 65,
        "confidence_score": 9.1,
        "production_ready": True,
    },
    {
        "title": "Generative Search",
        "description": "Generated answers from retrieved Weaviate objects using grouped_task and single_prompt.",
        "topic": "generative_search",
        "difficulty": "intermediate",
        "tags": ["generative", "llm", "rag"],
        "completed_at": dt(2026, 5, 11),
        "duration_minutes": 70,
        "confidence_score": 9.2,
        "production_ready": True,
    },
    {
        "title": "Text-to-Image Search",
        "description": "Used local CLIP embeddings and self-provided vectors to search images in Weaviate Cloud.",
        "topic": "image_search",
        "difficulty": "advanced",
        "tags": ["clip", "images", "self-provided-vectors"],
        "completed_at": dt(2026, 5, 13),
        "duration_minutes": 90,
        "confidence_score": 9.3,
        "production_ready": False,
    },
    {
        "title": "Focused RAG",
        "description": "Built a focused RAG collection from Weaviate notebooks and Markdown concept notes.",
        "topic": "rag",
        "difficulty": "advanced",
        "tags": ["rag", "chunking", "notebooks", "markdown"],
        "completed_at": dt(2026, 5, 15),
        "duration_minutes": 110,
        "confidence_score": 9.6,
        "production_ready": True,
    },
    {
        "title": "Named Vectors",
        "description": "Created multiple vector spaces for one object, such as title_vector and description_vector.",
        "topic": "named_vectors",
        "difficulty": "advanced",
        "tags": ["named-vectors", "schema", "embeddings"],
        "completed_at": dt(2026, 5, 17),
        "duration_minutes": 75,
        "confidence_score": 9.0,
        "production_ready": True,
    },
    {
        "title": "Cross-References",
        "description": "Connected review objects to product objects using reference properties.",
        "topic": "cross_references",
        "difficulty": "advanced",
        "tags": ["references", "relations", "schema"],
        "completed_at": dt(2026, 5, 19),
        "duration_minutes": 80,
        "confidence_score": 8.8,
        "production_ready": False,
    },
    {
        "title": "Multi-Tenancy",
        "description": "Created tenant-isolated data spaces inside one shared collection.",
        "topic": "multi_tenancy",
        "difficulty": "advanced",
        "tags": ["tenants", "saas", "isolation"],
        "completed_at": dt(2026, 5, 21),
        "duration_minutes": 85,
        "confidence_score": 9.2,
        "production_ready": True,
    },
    {
        "title": "Idempotent Imports",
        "description": "Used deterministic UUIDs and upsert logic to avoid duplicate imports.",
        "topic": "data_ingestion",
        "difficulty": "advanced",
        "tags": ["uuid", "upsert", "sync", "ingestion"],
        "completed_at": dt(2026, 5, 23),
        "duration_minutes": 95,
        "confidence_score": 9.4,
        "production_ready": True,
    },
]

In [7]:
result = milestones.data.insert_many(milestone_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [8]:
response = milestones.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "topic",
        "difficulty",
        "tags",
        "completed_at",
        "duration_minutes",
        "confidence_score",
        "production_ready",
    ],
)

for obj in response.objects:
    print(obj.properties)
    print("-" * 80)

{'duration_minutes': 70, 'confidence_score': 9.2, 'production_ready': True, 'title': 'Generative Search', 'topic': 'generative_search', 'completed_at': datetime.datetime(2026, 5, 11, 0, 0, tzinfo=datetime.timezone.utc), 'tags': ['generative', 'llm', 'rag'], 'difficulty': 'intermediate'}
--------------------------------------------------------------------------------
{'duration_minutes': 65, 'confidence_score': 9.1, 'production_ready': True, 'title': 'Hybrid Search', 'topic': 'hybrid_search', 'completed_at': datetime.datetime(2026, 5, 9, 0, 0, tzinfo=datetime.timezone.utc), 'difficulty': 'intermediate', 'tags': ['bm25', 'hybrid', 'semantic-search']}
--------------------------------------------------------------------------------
{'duration_minutes': 45, 'topic': 'docker', 'production_ready': False, 'title': 'Local Docker Weaviate', 'confidence_score': 7.8, 'completed_at': datetime.datetime(2026, 5, 1, 0, 0, tzinfo=datetime.timezone.utc), 'difficulty': 'beginner', 'tags': ['docker', 'loc

In [9]:
response = milestones.query.fetch_objects(
    filters=Filter.by_property("completed_at").greater_than(dt(2026, 5, 10)),
    limit=20,
    return_properties=[
        "title",
        "topic",
        "completed_at",
        "difficulty",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Completed at:", obj.properties["completed_at"])
    print("Difficulty:", obj.properties["difficulty"])
    print("-" * 80)

Title: Generative Search
Topic: generative_search
Completed at: 2026-05-11 00:00:00+00:00
Difficulty: intermediate
--------------------------------------------------------------------------------
Title: Text-to-Image Search
Topic: image_search
Completed at: 2026-05-13 00:00:00+00:00
Difficulty: advanced
--------------------------------------------------------------------------------
Title: Focused RAG
Topic: rag
Completed at: 2026-05-15 00:00:00+00:00
Difficulty: advanced
--------------------------------------------------------------------------------
Title: Named Vectors
Topic: named_vectors
Completed at: 2026-05-17 00:00:00+00:00
Difficulty: advanced
--------------------------------------------------------------------------------
Title: Cross-References
Topic: cross_references
Completed at: 2026-05-19 00:00:00+00:00
Difficulty: advanced
--------------------------------------------------------------------------------
Title: Multi-Tenancy
Topic: multi_tenancy
Completed at: 2026-05-21 0

In [10]:
date_range_filter = (
    Filter.by_property("completed_at").greater_or_equal(dt(2026, 5, 7))
    & Filter.by_property("completed_at").less_or_equal(dt(2026, 5, 17))
)

response = milestones.query.fetch_objects(
    filters=date_range_filter,
    limit=20,
    return_properties=[
        "title",
        "topic",
        "completed_at",
        "duration_minutes",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Completed at:", obj.properties["completed_at"])
    print("Duration:", obj.properties["duration_minutes"])
    print("-" * 80)

Title: Vector Search
Topic: vector_search
Completed at: 2026-05-07 00:00:00+00:00
Duration: 60
--------------------------------------------------------------------------------
Title: Hybrid Search
Topic: hybrid_search
Completed at: 2026-05-09 00:00:00+00:00
Duration: 65
--------------------------------------------------------------------------------
Title: Generative Search
Topic: generative_search
Completed at: 2026-05-11 00:00:00+00:00
Duration: 70
--------------------------------------------------------------------------------
Title: Text-to-Image Search
Topic: image_search
Completed at: 2026-05-13 00:00:00+00:00
Duration: 90
--------------------------------------------------------------------------------
Title: Focused RAG
Topic: rag
Completed at: 2026-05-15 00:00:00+00:00
Duration: 110
--------------------------------------------------------------------------------
Title: Named Vectors
Topic: named_vectors
Completed at: 2026-05-17 00:00:00+00:00
Duration: 75
----------------------

In [11]:
response = milestones.query.fetch_objects(
    filters=Filter.by_property("tags").contains_any(["rag", "chunking"]),
    limit=20,
    return_properties=[
        "title",
        "topic",
        "tags",
        "difficulty",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Tags:", obj.properties["tags"])
    print("Difficulty:", obj.properties["difficulty"])
    print("-" * 80)

Title: Generative Search
Topic: generative_search
Tags: ['generative', 'llm', 'rag']
Difficulty: intermediate
--------------------------------------------------------------------------------
Title: Focused RAG
Topic: rag
Tags: ['rag', 'chunking', 'notebooks', 'markdown']
Difficulty: advanced
--------------------------------------------------------------------------------


In [12]:
response = milestones.query.fetch_objects(
    filters=Filter.by_property("tags").contains_all(["rag", "chunking"]),
    limit=20,
    return_properties=[
        "title",
        "topic",
        "tags",
        "description",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Tags:", obj.properties["tags"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Title: Focused RAG
Topic: rag
Tags: ['rag', 'chunking', 'notebooks', 'markdown']
Description: Built a focused RAG collection from Weaviate notebooks and Markdown concept notes.
--------------------------------------------------------------------------------


In [13]:
advanced_production_filter = (
    Filter.by_property("difficulty").equal("advanced")
    & Filter.by_property("production_ready").equal(True)
)

response = milestones.query.fetch_objects(
    filters=advanced_production_filter,
    limit=20,
    return_properties=[
        "title",
        "topic",
        "difficulty",
        "production_ready",
        "confidence_score",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Production ready:", obj.properties["production_ready"])
    print("Confidence:", obj.properties["confidence_score"])
    print("-" * 80)

Title: Focused RAG
Topic: rag
Difficulty: advanced
Production ready: True
Confidence: 9.6
--------------------------------------------------------------------------------
Title: Named Vectors
Topic: named_vectors
Difficulty: advanced
Production ready: True
Confidence: 9.0
--------------------------------------------------------------------------------
Title: Multi-Tenancy
Topic: multi_tenancy
Difficulty: advanced
Production ready: True
Confidence: 9.2
--------------------------------------------------------------------------------
Title: Idempotent Imports
Topic: data_ingestion
Difficulty: advanced
Production ready: True
Confidence: 9.4
--------------------------------------------------------------------------------


In [14]:
rag_or_tenant_filter = (
    Filter.by_property("topic").equal("rag")
    | Filter.by_property("topic").equal("multi_tenancy")
)

response = milestones.query.fetch_objects(
    filters=rag_or_tenant_filter,
    limit=20,
    return_properties=[
        "title",
        "topic",
        "description",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Title: Focused RAG
Topic: rag
Description: Built a focused RAG collection from Weaviate notebooks and Markdown concept notes.
--------------------------------------------------------------------------------
Title: Multi-Tenancy
Topic: multi_tenancy
Description: Created tenant-isolated data spaces inside one shared collection.
--------------------------------------------------------------------------------


In [15]:
strong_advanced_filter = (
    Filter.by_property("production_ready").equal(True)
    & Filter.by_property("confidence_score").greater_or_equal(9.0)
    & Filter.by_property("difficulty").equal("advanced")
)

response = milestones.query.fetch_objects(
    filters=strong_advanced_filter,
    limit=20,
    return_properties=[
        "title",
        "topic",
        "difficulty",
        "confidence_score",
        "production_ready",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Confidence:", obj.properties["confidence_score"])
    print("Production ready:", obj.properties["production_ready"])
    print("-" * 80)

Title: Focused RAG
Topic: rag
Difficulty: advanced
Confidence: 9.6
Production ready: True
--------------------------------------------------------------------------------
Title: Named Vectors
Topic: named_vectors
Difficulty: advanced
Confidence: 9.0
Production ready: True
--------------------------------------------------------------------------------
Title: Multi-Tenancy
Topic: multi_tenancy
Difficulty: advanced
Confidence: 9.2
Production ready: True
--------------------------------------------------------------------------------
Title: Idempotent Imports
Topic: data_ingestion
Difficulty: advanced
Confidence: 9.4
Production ready: True
--------------------------------------------------------------------------------


In [16]:
recent_filter = Filter.by_property("completed_at").greater_or_equal(dt(2026, 5, 15))

response = milestones.query.near_text(
    query="features useful for production SaaS architecture",
    filters=recent_filter,
    limit=5,
    return_properties=[
        "title",
        "description",
        "topic",
        "completed_at",
        "tags",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Completed at:", obj.properties["completed_at"])
    print("Tags:", obj.properties["tags"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Distance: 0.6116939783096313
Title: Multi-Tenancy
Topic: multi_tenancy
Completed at: 2026-05-21 00:00:00+00:00
Tags: ['tenants', 'saas', 'isolation']
Description: Created tenant-isolated data spaces inside one shared collection.
--------------------------------------------------------------------------------
Distance: 0.6877350807189941
Title: Cross-References
Topic: cross_references
Completed at: 2026-05-19 00:00:00+00:00
Tags: ['references', 'relations', 'schema']
Description: Connected review objects to product objects using reference properties.
--------------------------------------------------------------------------------
Distance: 0.7484598159790039
Title: Idempotent Imports
Topic: data_ingestion
Completed at: 2026-05-23 00:00:00+00:00
Tags: ['uuid', 'upsert', 'sync', 'ingestion']
Description: Used deterministic UUIDs and upsert logic to avoid duplicate imports.
--------------------------------------------------------------------------------
Distance: 0.780015230178833
Title: N

In [17]:
tag_filter = Filter.by_property("tags").contains_any(
    ["rag", "semantic-search", "embeddings"]
)

response = milestones.query.near_text(
    query="retrieval and semantic search with generated answers",
    filters=tag_filter,
    limit=5,
    return_properties=[
        "title",
        "description",
        "topic",
        "tags",
        "confidence_score",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Tags:", obj.properties["tags"])
    print("Confidence:", obj.properties["confidence_score"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Distance: 0.48812901973724365
Title: Generative Search
Topic: generative_search
Tags: ['generative', 'llm', 'rag']
Confidence: 9.2
Description: Generated answers from retrieved Weaviate objects using grouped_task and single_prompt.
--------------------------------------------------------------------------------
Distance: 0.5269837379455566
Title: Hybrid Search
Topic: hybrid_search
Tags: ['bm25', 'hybrid', 'semantic-search']
Confidence: 9.1
Description: Combined BM25 keyword search with vector search using alpha.
--------------------------------------------------------------------------------
Distance: 0.5275466442108154
Title: Vector Search
Topic: vector_search
Tags: ['vectors', 'embeddings', 'semantic-search']
Confidence: 9.0
Description: Used near_text queries to search objects by semantic meaning.
--------------------------------------------------------------------------------
Distance: 0.7425272464752197
Title: Named Vectors
Topic: named_vectors
Tags: ['named-vectors', 'schema', 'e

In [18]:
hybrid_filter = (
    Filter.by_property("production_ready").equal(True)
    & Filter.by_property("confidence_score").greater_or_equal(8.8)
)

response = milestones.query.hybrid(
    query="RAG vectors cloud production",
    alpha=0.5,
    filters=hybrid_filter,
    limit=5,
    return_properties=[
        "title",
        "description",
        "topic",
        "difficulty",
        "confidence_score",
        "production_ready",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Confidence:", obj.properties["confidence_score"])
    print("Production ready:", obj.properties["production_ready"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Score: 1.0
Title: Focused RAG
Topic: rag
Difficulty: advanced
Confidence: 9.6
Production ready: True
Description: Built a focused RAG collection from Weaviate notebooks and Markdown concept notes.
--------------------------------------------------------------------------------
Score: 0.5134204030036926
Title: Named Vectors
Topic: named_vectors
Difficulty: advanced
Confidence: 9.0
Production ready: True
Description: Created multiple vector spaces for one object, such as title_vector and description_vector.
--------------------------------------------------------------------------------
Score: 0.2723132371902466
Title: Generative Search
Topic: generative_search
Difficulty: intermediate
Confidence: 9.2
Production ready: True
Description: Generated answers from retrieved Weaviate objects using grouped_task and single_prompt.
--------------------------------------------------------------------------------
Score: 0.09078104048967361
Title: Multi-Tenancy
Topic: multi_tenancy
Difficulty: advan

In [19]:
def search_timeline(
    query: str,
    *,
    from_date: datetime,
    to_date: datetime,
    tags: list[str] | None = None,
    production_ready: bool | None = None,
    limit: int = 5,
) -> None:
    filters = (
        Filter.by_property("completed_at").greater_or_equal(from_date)
        & Filter.by_property("completed_at").less_or_equal(to_date)
    )

    if tags:
        filters = filters & Filter.by_property("tags").contains_any(tags)

    if production_ready is not None:
        filters = filters & Filter.by_property("production_ready").equal(production_ready)

    response = milestones.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "title",
            "description",
            "topic",
            "difficulty",
            "tags",
            "completed_at",
            "confidence_score",
            "production_ready",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("FROM:", from_date)
    print("TO:", to_date)
    print("TAGS:", tags)
    print("PRODUCTION READY:", production_ready)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Topic:", obj.properties["topic"])
        print("Difficulty:", obj.properties["difficulty"])
        print("Tags:", obj.properties["tags"])
        print("Completed at:", obj.properties["completed_at"])
        print("Confidence:", obj.properties["confidence_score"])
        print("Production ready:", obj.properties["production_ready"])
        print("Description:", obj.properties["description"])
        print("-" * 80)

In [20]:
search_timeline(
    "production-ready Weaviate features for SaaS and RAG applications",
    from_date=dt(2026, 5, 10),
    to_date=dt(2026, 5, 25),
    production_ready=True,
)

QUERY: production-ready Weaviate features for SaaS and RAG applications
FROM: 2026-05-10 00:00:00+00:00
TO: 2026-05-25 00:00:00+00:00
TAGS: None
PRODUCTION READY: True
--------------------------------------------------------------------------------
Distance: 0.5321423411369324
Title: Focused RAG
Topic: rag
Difficulty: advanced
Tags: ['rag', 'chunking', 'notebooks', 'markdown']
Completed at: 2026-05-15 00:00:00+00:00
Confidence: 9.6
Production ready: True
Description: Built a focused RAG collection from Weaviate notebooks and Markdown concept notes.
--------------------------------------------------------------------------------
Distance: 0.561297595500946
Title: Generative Search
Topic: generative_search
Difficulty: intermediate
Tags: ['generative', 'llm', 'rag']
Completed at: 2026-05-11 00:00:00+00:00
Confidence: 9.2
Production ready: True
Description: Generated answers from retrieved Weaviate objects using grouped_task and single_prompt.
----------------------------------------------

In [21]:
search_timeline(
    "semantic search and embeddings",
    from_date=dt(2026, 5, 1),
    to_date=dt(2026, 5, 20),
    tags=["embeddings", "semantic-search"],
)

QUERY: semantic search and embeddings
FROM: 2026-05-01 00:00:00+00:00
TO: 2026-05-20 00:00:00+00:00
TAGS: ['embeddings', 'semantic-search']
PRODUCTION READY: None
--------------------------------------------------------------------------------
Distance: 0.3917340040206909
Title: Vector Search
Topic: vector_search
Difficulty: intermediate
Tags: ['vectors', 'embeddings', 'semantic-search']
Completed at: 2026-05-07 00:00:00+00:00
Confidence: 9.0
Production ready: True
Description: Used near_text queries to search objects by semantic meaning.
--------------------------------------------------------------------------------
Distance: 0.45802223682403564
Title: Hybrid Search
Topic: hybrid_search
Difficulty: intermediate
Tags: ['bm25', 'hybrid', 'semantic-search']
Completed at: 2026-05-09 00:00:00+00:00
Confidence: 9.1
Production ready: True
Description: Combined BM25 keyword search with vector search using alpha.
--------------------------------------------------------------------------------

In [22]:
search_timeline(
    "data ingestion and synchronization",
    from_date=dt(2026, 5, 20),
    to_date=dt(2026, 5, 30),
    tags=["upsert", "sync", "ingestion"],
    production_ready=True,
)

QUERY: data ingestion and synchronization
FROM: 2026-05-20 00:00:00+00:00
TO: 2026-05-30 00:00:00+00:00
TAGS: ['upsert', 'sync', 'ingestion']
PRODUCTION READY: True
--------------------------------------------------------------------------------
Distance: 0.44975584745407104
Title: Idempotent Imports
Topic: data_ingestion
Difficulty: advanced
Tags: ['uuid', 'upsert', 'sync', 'ingestion']
Completed at: 2026-05-23 00:00:00+00:00
Confidence: 9.4
Production ready: True
Description: Used deterministic UUIDs and upsert logic to avoid duplicate imports.
--------------------------------------------------------------------------------


In [23]:
filters = (
    Filter.by_property("completed_at").greater_or_equal(dt(2026, 5, 10))
    & Filter.by_property("difficulty").equal("advanced")
    & Filter.by_property("production_ready").equal(True)
)

response = milestones.generate.near_text(
    query="advanced Weaviate features useful for real applications",
    filters=filters,
    grouped_task=(
        "Summarize the retrieved learning milestones. "
        "Explain which Weaviate concepts are most useful for building real applications. "
        "Use only the retrieved milestones."
    ),
    limit=6,
    return_properties=[
        "title",
        "description",
        "topic",
        "tags",
        "completed_at",
        "confidence_score",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "|",
        obj.properties["topic"],
        "|",
        obj.properties["completed_at"],
    )

The retrieved learning milestones highlight several key achievements in utilizing Weaviate for advanced applications. Here is a summary of each milestone:

1. **Focused RAG (May 15, 2026)**: Developed a specialized Retrieval-Augmented Generation (RAG) collection by leveraging Weaviate notebooks and Markdown notes. This milestone reflects a strong understanding of data structuring and integration in Weaviate, crucial for creating applications that utilize RAG techniques effectively.

2. **Named Vectors (May 17, 2026)**: Established multiple vector spaces for a single object, enabling distinct representations like title_vector and description_vector. This achievement emphasizes the flexibility of Weaviate’s schema management and its ability to handle complex embeddings, which are essential for applications that require nuanced semantic searches.

3. **Multi-Tenancy (May 21, 2026)**: Created isolated data spaces for different tenants within a shared collection, facilitating a Software as 

In [24]:
client.close()